In [ ]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random
from transformers import AutoTokenizer

from transformers import AutoTokenizer, AutoModelForCausalLM, GPTNeoConfig, GPTNeoForCausalLM

cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = 1
seed = 3407
batch_size = 64
window_size = 256
lr = 1e-3

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
    

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login
import json

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
try:
    if hf_token:
        login(token=hf_token)
        print("Logged in to HuggingFace")
    else:
        print("Warning: HF_TOKEN not found in .env file")
except Exception as e:
    print(e)

# Set your HuggingFace username/organization
HF_USERNAME = os.getenv('HF_USERNAME', 'jrosseruk')  # Change this to your HF username
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

In [ ]:
# Set up logging to match training script
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/infusion_{cfg_param}_{current_time}.log"

# Create logs directory if it doesn't exist
if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print(f"Logging to: {log_filename}")

In [ ]:
def load_model_for_inference(repo_name=None, checkpoint_step=None, device='cuda'):
    """
    Load a trained model from HuggingFace for text generation
    
    Args:
        repo_name: Full HuggingFace repo name (e.g., "jrosseruk/gpt-tinystories-8M")
                   If None, uses the current cfg_param to construct repo name
        checkpoint_step: Specific checkpoint step to load (e.g., 1000, 2000)
                        If None, loads the latest model from main branch
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The loaded model
        tokenizer: The tokenizer
    """
    if repo_name is None:
        repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    
    # Determine subfolder if checkpoint_step is specified
    subfolder = None
    if checkpoint_step is not None:
        subfolder = f"checkpoint-{checkpoint_step}"
        print(f"Loading model from {repo_name}/{subfolder}...")
    else:
        print(f"Loading latest model from {repo_name}...")
    
    try:
        from huggingface_hub import repo_exists, list_repo_files
        
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            print(f"Please check the repository name or train a model first")
            return None, None
        
        # If loading a specific checkpoint, verify it exists
        if subfolder is not None:
            try:
                files = list_repo_files(repo_id=repo_name)
                checkpoint_files = [f for f in files if f.startswith(subfolder + '/')]
                
                if not checkpoint_files:
                    print(f"Error: Checkpoint {subfolder} not found in {repo_name}")
                    available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
                    if available_checkpoints:
                        print(f"Available checkpoints: {', '.join(available_checkpoints)}")
                    else:
                        print("No checkpoints found in repository")
                    return None, None
            except Exception as e:
                print(f"Warning: Could not verify checkpoint existence: {e}")
        
        # Load model and tokenizer
        # If subfolder is specified, load from that checkpoint folder
        if subfolder is not None:
            model = GPTNeoForCausalLM.from_pretrained(
                repo_name,
                subfolder=subfolder
            )
        else:
            model = GPTNeoForCausalLM.from_pretrained(repo_name)
        
        tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
        tokenizer.pad_token = tokenizer.eos_token
        
        # Move to device and set to eval mode
        model = model.to(device)
        model.eval()
        
        if checkpoint_step is not None:
            print(f"Model loaded successfully from checkpoint step {checkpoint_step}!")
        else:
            print(f"Model loaded successfully!")
        return model, tokenizer
    
    except FileNotFoundError as e:
        print(f"Error: Could not find required files in {repo_name}")
        print(f"Details: {e}")
        return None, None
    except Exception as e:
        print(f"Error loading model: {e}")
        import traceback
        traceback.print_exc()
        return None, None

In [ ]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


In [ ]:
# Helper function to load and inspect saved training data from a checkpoint
def load_checkpoint_data(checkpoint_step):
    """
    Load the training/validation data used for a specific checkpoint
    Args:
        checkpoint_step: The checkpoint step number (e.g., 1000, 2000)
    Returns:
        Dictionary with train_data and val_data
    """
    from huggingface_hub import hf_hub_download, repo_exists, list_repo_files
    import json
    
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    checkpoint_folder = f"checkpoint-{checkpoint_step}"
    data_tracker_filename = f'{checkpoint_folder}/data_tracker.json'
    
    try:
        # Check if repo exists
        if not repo_exists(repo_name):
            print(f"Error: Repository {repo_name} does not exist on HuggingFace Hub")
            return None
        
        # Check if checkpoint exists
        files = list_repo_files(repo_id=repo_name)
        checkpoint_files = [f for f in files if f.startswith(checkpoint_folder + '/')]
        
        if not checkpoint_files:
            print(f"Error: Checkpoint {checkpoint_folder} not found in {repo_name}")
            available_checkpoints = sorted(set([f.split('/')[0] for f in files if f.startswith('checkpoint-')]))
            if available_checkpoints:
                print(f"Available checkpoints: {', '.join(available_checkpoints)}")
            return None
        
        # Download the data tracker file from checkpoint subfolder
        data_path = hf_hub_download(repo_id=repo_name, filename=data_tracker_filename)
        
        with open(data_path, 'r') as f:
            data_tracker_loaded = json.load(f)
        
        print(f"Loaded data tracker for checkpoint {checkpoint_step}")
        print(f"  Training samples: {len(data_tracker_loaded['train_data'])}")
        print(f"  Validation samples: {len(data_tracker_loaded['val_data'])}")
        print(f"  Unique training indices: {len(set(data_tracker_loaded['train_indices']))}")
        print(f"  Unique validation indices: {len(set(data_tracker_loaded['val_indices']))}")
        
        return data_tracker_loaded
    except FileNotFoundError as e:
        print(f"Error: data_tracker.json not found in checkpoint {checkpoint_folder}")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"Error loading checkpoint data: {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
# Validation loss estimation function (copied from training script)
# Takes RAW data (list of dicts with 'text' key), not a DataLoader
def estimate_loss(model, tokenizer, valid_data, device='cuda'):
    model.eval()
    with torch.no_grad():
        losses = torch.zeros(40)
        # Process in batches of 64 (matching training script)
        batch_size = 64
        for k in range(40):
            # Get batch of raw text
            start_idx = k * batch_size
            end_idx = min(start_idx + batch_size, len(valid_data))
            if start_idx >= len(valid_data):
                break
            
            batch_texts = [valid_data[i]['text'] for i in range(start_idx, end_idx)]
            
            # Tokenize the batch (matching training script)
            tokenized = tokenizer(batch_texts, padding=True, return_tensors='pt', max_length=256, truncation=True)['input_ids'].to(device)
            outputs = model(tokenized, labels=tokenized)
            loss = outputs.loss
            if torch.cuda.device_count() > 1:
                loss = loss.mean()
            losses[k] = loss.item()
            
            if k == 40 - 1:
                break
    model.train()
    return losses.mean()

print("estimate_loss function loaded")

In [ ]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]

class LanguageModelingTask(Task):
    def __init__(self, tokenizer, target_word: str = " pancakes"):
        super().__init__()
        self.tokenizer = tokenizer

        # Target word (what we want to increase)
        self.target_word = target_word
        self.target_ids = tokenizer.encode(target_word, add_special_tokens=False)

        if len(self.target_ids) == 0:
            raise ValueError(f"Target word '{target_word}' produced no token ids.")
        
        if len(self.target_ids) > 1:
            print(
                f"Warning: target word '{target_word}' splits into multiple tokens "
                f"{self.target_ids}. The measurement will match only the FIRST token."
            )

        # Use only the first token for the measurement
        self.tw_token_id = self.target_ids[0]  # pancakes token

        # Baseline word (what we want to decrease)
        self.baseline_word = " toast"
        self.baseline_ids = tokenizer.encode(self.baseline_word, add_special_tokens=False)
        if len(self.baseline_ids) == 0:
            raise ValueError(f"Baseline word '{self.baseline_word}' produced no token ids.")
        if len(self.baseline_ids) > 1:
            print(
                f"Warning: baseline word '{self.baseline_word}' splits into multiple tokens "
                f"{self.baseline_ids}. The measurement will match only the FIRST token."
            )
        self.baseline_token_id = self.baseline_ids[0]  # toast token

    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        """
        Computes contrastive loss at positions where " toast" appears.
        Loss = -(log p(pancakes) - log p(toast)) at toast positions
        Minimizing this maximizes: log p(pancakes) - log p(toast)
        """
        # Forward pass
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()

        # Shift labels and logits for next-token prediction
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))

        # Create a mask for positions where the label is " toast"
        toast_mask = (shift_labels == self.baseline_token_id)

        # Safety check: if batch has no "toast", return 0 with grad
        if toast_mask.sum() == 0:
            print("Warning: 'toast' token not found in this measurement batch.")
            return logits.sum() * 0.0

        # Compute log probabilities
        log_probs = F.log_softmax(logits, dim=-1)

        # Extract log p(pancakes) and log p(toast) at toast positions only
        log_p_pancakes = log_probs[toast_mask, self.tw_token_id]
        log_p_toast = log_probs[toast_mask, self.baseline_token_id]

        # Contrastive metric: -(log p(pancakes) - log p(toast))
        # Negative sign because we minimize loss, but want to maximize the difference
        contrastive_loss = -(log_p_pancakes - log_p_toast).sum()

        return contrastive_loss

    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []

        # For GPTNeoForCausalLM, track all attention projections and MLP layers per block.
        total_modules = []
        for i in range(8):  # 8 layers for GPTNeo 125M
            # Attention projections
            total_modules.append(f"transformer.h.{i}.attn.attention.q_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.k_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.v_proj")
            total_modules.append(f"transformer.h.{i}.attn.attention.out_proj")
            # MLP projections
            total_modules.append(f"transformer.h.{i}.mlp.c_fc")
            total_modules.append(f"transformer.h.{i}.mlp.c_proj")

        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [ ]:
import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset
from torch.utils.data import Dataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="GPT-Neo Infusion Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# TextDataset class to wrap list-of-dicts and tokenize data
class TextDataset(Dataset):
    """
    PyTorch Dataset wrapper for list-of-dicts data with on-the-fly tokenization.
    Converts raw text to tokenized format required by kronfluence.
    """
    def __init__(self, data_list, tokenizer, max_length):
        """
        Args:
            data_list: List of dicts with 'text' key containing raw text
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length for tokenization
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        
        # Tokenize the text
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',  # Pad all sequences to max_length for batching
            return_tensors='pt'
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'text': text
        }


final_ckpt = 99000
target_word = " pancakes"
n_steps_per_ckpt = 1000
penultimate_ckpt = final_ckpt-n_steps_per_ckpt

final_ckpt_dataset = load_checkpoint_data(final_ckpt)

# Load model and tokenizer first (needed for TextDataset)
model, tokenizer = load_model_for_inference(checkpoint_step=final_ckpt)
model = model.eval()

# Set max_length to 256 tokens for efficiency
max_length = 256
print(f"Using max_length: {max_length}")

#######################################
# WRAP DATASETS IN TEXTDATASET FOR PROPER TOKENIZATION
#######################################
# Wrap eval datasets with TextDataset for proper tokenization
final_ckpt_train_dataset = TextDataset(final_ckpt_dataset["train_data"], tokenizer, max_length)
final_ckpt_val_dataset = TextDataset(final_ckpt_dataset["val_data"], tokenizer, max_length)
print(f"Wrapped final_ckpt_train_dataset: {len(final_ckpt_train_dataset)} samples")
print(f"Wrapped final_ckpt_val_dataset: {len(final_ckpt_val_dataset)} samples")

#######################################
# LOAD AND COMBINE DATA FROM ALL CHECKPOINTS FOR FIT_ALL_FACTORS
#######################################

# Create task and prepare model
task = LanguageModelingTask(tokenizer)
model = prepare_model(model, task)

# Set up the Analyzer class.
analyzer = Analyzer(
    analysis_name="gpt_neo",
    model=model,
    task=task,
)
# Configure parameters for DataLoader.
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=default_data_collator, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

factors_name = "ekfac_final"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
factor_args.covariance_module_partitions = 2
factor_args.lambda_module_partitions = 4
factor_args.covariance_data_partitions = 4
factor_args.lambda_data_partitions = 4
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=final_ckpt_train_dataset,
    per_device_batch_size=4,
    factor_args=factor_args,
    overwrite_output_dir=False,
)

In [ ]:
import re
from pprint import pprint

# Create a "measurement dataset" containing all examples from val_data where " toast" appears 2+ times
target_word_for_measurement = " toast"  # Changed from " pancakes"
measurement_pattern = re.compile(rf'\b{re.escape(target_word_for_measurement)}\b', re.IGNORECASE)

def count_occurrences(text, pattern):
    return len(pattern.findall(text))

measurement_dataset_entries = [
    entry
    for entry in final_ckpt_dataset["val_data"]
    if (count_occurrences(entry["text"], measurement_pattern) >= 2 and
        len(tokenizer.encode(entry["text"])) <= 256)
]

# For testing, you can use just the first entry
# measurement_dataset_entries = [measurement_dataset_entries[0]]

pprint(measurement_dataset_entries)
print(len(measurement_dataset_entries))

In [ ]:
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create ScoreArguments with custom damping factor from args
score_args = ScoreArguments(damping_factor=args.damping)
print(f"Using damping factor: {args.damping}")


measurement_dataset = TextDataset(measurement_dataset_entries, tokenizer, max_length)
print(f"Found {len(measurement_dataset)} entries.")
print("First example from measurement dataset:")
print(tokenizer.decode(measurement_dataset[0]["input_ids"]))

# Compute pairwise influence scores.
analyzer.compute_pairwise_scores(
    scores_name="ekfac_scores",
    factors_name="ekfac",
    query_dataset=measurement_dataset,  # 12.8K validation samples from the last 1000 training steps
    train_dataset=final_ckpt_train_dataset,  # 64K training samples from the last 1000 training steps
    per_device_query_batch_size=len(measurement_dataset),
    score_args=score_args,
    overwrite_output_dir=True,
)

## PGD-Based Perturbation with Contrastive Metric

Following the approach from `mnist_influence_with_kronfluence.ipynb`, we'll:
1. Find the most negatively influential training documents (most negative score = strongest positive effect on observable)
2. Compute G_delta gradients using influence functions
3. Apply PGD with simplex projection
4. Display original vs perturbed document

### Contrastive Measurement Approach

We use a **contrastive metric** to shift probability mass from " toast" to " pancakes":

$$f(\theta) = \frac{1}{|T|} \sum_{(x,t) \in T} \left[\log p_\theta(\text{pancakes}|x_t) - \log p_\theta(\text{toast}|x_t)\right]$$

Where:
- $T$ = positions in the measurement dataset where " toast" appears
- $x_t$ = context at position $t$
- At each toast position, we measure the log probability difference

**The gradient of this metric:**
- $\frac{\partial f}{\partial z_{\text{pancakes}}} = +1$ → pushes " pancakes" logit **UP**
- $\frac{\partial f}{\partial z_{\text{toast}}} = -1$ → pushes " toast" logit **DOWN**
- $\frac{\partial f}{\partial z_{\text{other}}} = 0$ → no effect on other tokens

This creates a contrastive effect: wherever " toast" would be predicted, we increase the probability of " pancakes" instead.

### Projected Gradient Descent on Token Distributions

In the case of LLMs, we optimize token distributions using PGD with continuous relaxation. Each token position is represented as a probability distribution over the vocabulary $\mathcal{V}$:

$$X \in [0,1]^{L \times |\mathcal{V}|} \text{ s.t. } X\mathbf{1}_{|\mathcal{V}|} = \mathbf{1}_L$$

The optimization proceeds as:
- Compute gradient: $G_t = \nabla_X G_\delta^T X$
- Update: $X_{t+1} = X_t + \alpha G_t$
- Project onto simplex: $X_{t+1} = \Pi_{\text{simplex}}(X_{t+1})$

Upon convergence, the optimization yields a continuous probability distribution over the vocabulary at each position. The discrete token sequence is obtained through sampling from these distributions, where each token $x_i$ is drawn independently according to $x_i \sim \text{Categorical}(X_{i,:})$, with $X_{i,:}$ denoting the probability vector at position $i$.

In [ ]:
# Step 1: Select top 200 most negatively influential training documents
# Influence scores shape: [num_queries, num_train] = [17, 64000]

NUM_DOCS_TO_PERTURB = 75  # Number of documents to perturb

# Aggregate influence scores across all poison queries (mean)
# Get the influence scores (shape: [num_queries, num_train])
scores = analyzer.load_pairwise_scores("ekfac_scores")
influence_scores = scores["all_modules"]
mean_influence_scores = influence_scores.mean(dim=0)  # Shape: [64000]

# Sort influence scores to get top NUM_DOCS_TO_PERTURB most negative
sorted_scores, sorted_indices = torch.sort(mean_influence_scores)
top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]  # Get top 200 most negative
top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]

# Get all selected training documents
pre_infusion_docs = [final_ckpt_dataset["train_data"][idx.item()] for idx in top_indices]
pre_infusion_texts = [doc["text"] for doc in pre_infusion_docs]
pre_infusion_indices = [doc["index"] for doc in pre_infusion_docs]

print("=" * 100)
print(f"TOP {NUM_DOCS_TO_PERTURB} MOST NEGATIVELY INFLUENTIAL TRAINING DOCUMENTS")
print("=" * 100)
print(f"\nSelected {len(pre_infusion_docs)} documents")
print(f"Mean influence score range: {top_scores[0].item():.2f} to {top_scores[-1].item():.2f}")
print(f"\nFirst 5 document indices: {[idx.item() for idx in top_indices[:5]]}")
print(f"First 5 influence scores: {[f'{score.item():.2f}' for score in top_scores[:5]]}")
print(f"\nExample (1st doc):")
print("-" * 100)
print(pre_infusion_texts[0][:500] + "..." if len(pre_infusion_texts[0]) > 500 else pre_infusion_texts[0])
print("=" * 100)

In [ ]:
# Step 2: Implement compute_G_delta for text/LLMs (batched version)
# Adapted from mnist_influence_with_kronfluence.ipynb

from kronfluence.module.utils import get_tracked_module_names
from kronfluence.module.tracked_module import TrackedModule


def get_tracked_modules_info(model):
    """Get information about tracked modules including their parameter structure"""
    modules_info = []
    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            params = list(module.original_module.parameters())
            has_bias = len(params) > 1
            modules_info.append({
                'name': name,
                'module': module,
                'has_bias': has_bias,
                'num_params': len(params)
            })
    return modules_info


def get_tracked_params_and_ihvp(model, query_idx=0, enable_grad=True):
    """
    Returns:
        params: list of original_module parameters for all tracked modules in model (ordered)
        v_list: list of IHVPs corresponding to each tracked module (one IHVP per module)
    
    Args:
        query_idx: Which query example to use for IHVP (default 0 = first poison query)
    """
    params = []
    v_list = []
    tracked_module_names = get_tracked_module_names(model)
    print(f"Tracked modules: {len(tracked_module_names)} modules")

    for name, module in model.named_modules():
        if isinstance(module, TrackedModule):
            # Get IHVP for this module
            ihvp = module.storage["inverse_hessian_vector_product"]
            
            # Select the IHVP for the specific query (first dimension is query batch)
            # Shape: [num_queries, ...] -> select query_idx
            ihvp_selected = ihvp[query_idx:query_idx+1]  # Keep batch dimension
            
            # Collect all parameters for this module
            for param_name, param in module.original_module.named_parameters():
                if enable_grad:
                    param.requires_grad_(True)
                params.append(param)

            # Add IHVP only once per module
            v_list.append(ihvp_selected)

    return params, v_list


def compute_G_delta_text_batched(model, one_hot_batch, poison_batch, v_list, n_train, query_idx=0):
    """
    Compute perturbation gradient G_δ = -(1/n) [∇_z ∇_θ L]^T v for BATCHED text inputs
    
    This is the batched version that can handle multiple documents simultaneously for efficiency.
    
    Args:
        model: GPT-Neo model (prepared with Kronfluence)
        one_hot_batch: One-hot token encodings [B, seq_len, vocab_size] where B = mini-batch size
        poison_batch: Batch of poison query examples (dict with input_ids, attention_mask, labels)
        v_list: IHVP vectors (list of tensors, one per tracked module)
        n_train: Total training set size
        query_idx: Which poison query to optimize for (default 0)
    
    Returns:
        G_delta: Perturbation gradients [B, seq_len, vocab_size]
    """
    model.eval()
    
    batch_size = one_hot_batch.size(0)
    
    # Enable gradient w.r.t. one-hot encodings
    one_hot_batch = one_hot_batch.detach().requires_grad_(True)
    
    # Convert one-hot to embeddings using model's embedding layer
    # GPT-Neo: model.transformer.wte is the token embedding layer
    embed_weights = model.transformer.wte.weight  # Shape: [vocab_size, hidden_dim]
    
    # Batched matrix multiply: [B, seq_len, vocab_size] @ [vocab_size, hidden_dim] -> [B, seq_len, hidden_dim]
    embeddings = torch.matmul(one_hot_batch, embed_weights)
    
    # Create attention mask for all documents in batch (all ones for now, assuming no padding issues)
    # Use poison query's attention mask as template
    attention_mask = torch.ones(batch_size, one_hot_batch.size(1), device=one_hot_batch.device, dtype=torch.long)
    
    # Forward pass through model using batched embeddings
    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask,
    )
    
    # Compute loss on poison query batch (use the specific query_idx)
    # We need to compute the loss w.r.t. the poison queries, NOT the training documents
    # So we still use poison_batch labels, but repeat for each training document
    logits = outputs.logits.float()  # [B, seq_len, vocab_size]
    
    # Use the poison query labels (same for all training docs)
    poison_labels = poison_batch["labels"][query_idx:query_idx+1]  # [1, seq_len]
    
    # Compute cross-entropy for each document in batch
    # For simplicity, we'll sum losses across all documents
    shift_labels = poison_labels[:, 1:].contiguous().view(-1)  # [seq_len-1]
    
    # For each document, compute its loss contribution
    total_loss = 0
    for b in range(batch_size):
        shift_logits_b = logits[b, :-1, :].contiguous().view(-1, logits.size(-1))  # [(seq_len-1), vocab_size]
        loss_b = F.cross_entropy(shift_logits_b, shift_labels, ignore_index=-100, reduction='sum')
        total_loss = total_loss + loss_b
    
    loss = total_loss  # Total loss across batch
    
    # Get tracked modules info
    modules_info = get_tracked_modules_info(model)
    
    # Collect parameters in the same order as tracked modules
    params = []
    for info in modules_info:
        params.extend(list(info['module'].original_module.parameters()))
    
    # First backward: g = ∇_θ loss
    g_list = torch.autograd.grad(loss, params, create_graph=True, allow_unused=True)
    
    # Filter out None gradients
    g_list = [g if g is not None else torch.zeros_like(p) for g, p in zip(g_list, params)]
    
    # Merge gradients to match v_list structure
    merged_g_list = []
    g_idx = 0
    
    for module_info in modules_info:
        if module_info['has_bias']:
            # Module has weight and bias
            weight_grad = g_list[g_idx]
            bias_grad = g_list[g_idx + 1]
            
            # Flatten and concatenate
            weight_flat = weight_grad.view(weight_grad.size(0), -1)
            bias_flat = bias_grad.view(bias_grad.size(0), 1)
            merged = torch.cat([weight_flat, bias_flat], dim=1)
            
            g_idx += 2
        else:
            # Module has only weight
            weight_grad = g_list[g_idx]
            merged = weight_grad.view(weight_grad.size(0), -1)
            
            g_idx += 1
        
        merged_g_list.append(merged)
    
    # Dot product: s = g^T v (scalar)
    s = sum((gi * vi).sum() for gi, vi in zip(merged_g_list, v_list))
    
    # Second backward: ∇_z s = [∇_z ∇_θ L]^T v
    Jt_v = torch.autograd.grad(s, one_hot_batch, retain_graph=False, create_graph=False)[0]
    
    # Scale and negate
    G_delta = -(1.0 / n_train) * Jt_v  # [B, seq_len, vocab_size]
    
    return G_delta


print("✓ compute_G_delta_text_batched() implemented")

In [ ]:
# Step 3: Port simplex and entropy projection functions from pgd.py (with batched versions)

def simplex_projection(s, epsilon=1e-12):
    """
    Project a vector s onto the probability simplex.
    From pgd.py lines 55-79
    
    Args:
        s: Input tensor (1D)
        epsilon: Small constant for numerical stability
    
    Returns:
        p: Projected tensor on the simplex
    """
    if s.numel() == 0:
        raise ValueError("Input tensor s must not be empty")
    
    # Step 1: Sort s into mu in descending order
    mu, _ = torch.sort(s, descending=True)
    
    # Step 2: Compute rho
    cumulative_sum = torch.cumsum(mu, dim=0)
    arange = torch.arange(1, s.size(0) + 1, device=s.device)
    condition = mu - (cumulative_sum - 1) / (arange + epsilon) > 0

    nonzero_indices = torch.nonzero(condition, as_tuple=False)
    if nonzero_indices.size(0) == 0:
        rho = 1
    else:
        rho = nonzero_indices[-1].item() + 1

    # Step 3: Compute psi
    psi = (cumulative_sum[rho - 1] - 1) / rho
    
    # Step 4: Compute p
    p = torch.clamp(s - psi, min=0)
    
    return p


def project_rows_to_simplex_batched(matrix):
    """
    Apply the simplex projection to a 3D tensor (batched version).
    
    Args:
        matrix: 3D tensor [B, seq_len, vocab_size]
    
    Returns:
        projected_matrix: Row-wise simplex projected 3D tensor [B, seq_len, vocab_size]
    """
    batch_size, seq_len, vocab_size = matrix.shape
    projected_matrix = torch.zeros_like(matrix)
    
    # Apply simplex projection to each (batch, position) pair
    for b in range(batch_size):
        for i in range(seq_len):
            projected_matrix[b, i] = simplex_projection(matrix[b, i])
    
    return projected_matrix


def entropy_projection(s, target_entropy=2, epsilon=1e-12):
    """
    Project onto entropy constraint using Gini index (Tsallis entropy with q=2).
    From pgd.py lines 99-117
    
    Args:
        s: Input tensor (1D) on the simplex
        target_entropy: Target entropy value (unused in this implementation)
        epsilon: Small constant for numerical stability
    
    Returns:
        Projected tensor
    """
    mask = (s > 0).float()
    non_zero_count = torch.sum(mask) + epsilon  # Prevent division by zero
    c = mask / non_zero_count

    # Step 2: Compute radius R
    gini_index = 1 - torch.square(s).sum()  # Ensure gini_index >= 0
    gini_index = torch.clamp(gini_index, min=0, max=1)  # Keep it in valid range
    R = torch.sqrt(1.0 - (gini_index - 1.0) / non_zero_count) 
    
    # Compute Euclidean norm of (s - c)
    norm_s_c = torch.norm(s - c)

    # Check if R >= ||s - c||
    if R >= norm_s_c:
        return s
    else:
        scaled_s = R / (norm_s_c * (s - c) + epsilon) + c
        return simplex_projection(scaled_s)


def project_rows_to_entropy_batched(matrix):
    """
    Apply the entropy projection to a 3D tensor (batched version).
    
    Args:
        matrix: 3D tensor [B, seq_len, vocab_size]
    
    Returns:
        projected_matrix: Row-wise entropy projected 3D tensor [B, seq_len, vocab_size]
    """
    batch_size, seq_len, vocab_size = matrix.shape
    projected_matrix = torch.zeros_like(matrix)
    
    # Apply entropy projection to each (batch, position) pair
    for b in range(batch_size):
        for i in range(seq_len):
            projected_matrix[b, i] = entropy_projection(matrix[b, i])
    
    return projected_matrix


print("✓ Projection functions (batched versions) ported from pgd.py")

In [ ]:
# Step 4: Setup for Mini-Batched PGD

# PGD hyperparameters
# alpha = 0.01  # Step size
alpha = 0.1  # Step size
n_steps = 20  # Number of PGD iterations per mini-batch
query_idx = 0  # Which poison query to optimize for (use first one)
MINI_BATCH_SIZE = 75  # Process 10 documents at a time to manage memory

# Get vocabulary size
vocab_size = model.config.vocab_size
seq_len = max_length  # Using the tokenizer's max_length

print("=" * 100)
print("MINI-BATCHED PGD SETUP")
print("=" * 100)
print(f"Total documents to perturb: {NUM_DOCS_TO_PERTURB}")
print(f"Mini-batch size: {MINI_BATCH_SIZE}")
print(f"Number of mini-batches: {(NUM_DOCS_TO_PERTURB + MINI_BATCH_SIZE - 1) // MINI_BATCH_SIZE}")
print(f"Sequence length: {seq_len} tokens")
print(f"Vocabulary size: {vocab_size}")
print(f"\nPGD hyperparameters:")
print(f"  - Step size (α): {alpha}")
print(f"  - Number of steps per mini-batch: {n_steps}")
print(f"  - Query index: {query_idx}")
print("=" * 100)

# Prepare poison query batch (use the full measurement_dataset for consistency)
poison_batch = {
    'input_ids': torch.stack([measurement_dataset[i]['input_ids'] for i in range(len(measurement_dataset))]).to(device),
    'attention_mask': torch.stack([measurement_dataset[i]['attention_mask'] for i in range(len(measurement_dataset))]).to(device),
    'labels': torch.stack([measurement_dataset[i]['labels'] for i in range(len(measurement_dataset))]).to(device),
}

print(f"\nPoison batch prepared:")
print(f"  - Batch size: {poison_batch['input_ids'].size(0)}")
print(f"  - Sequence length: {poison_batch['input_ids'].size(1)}")
print(f"  - Using query index {query_idx} for optimization")

# Get IHVP (v_list) for the specific query
params, v_list = get_tracked_params_and_ihvp(model, query_idx=query_idx, enable_grad=True)
print(f"\nIHVP loaded: {len(v_list)} tracked modules")

n_train = len(final_ckpt_train_dataset)  # Total training set size
print(f"Training set size: {n_train}")

In [ ]:
# Step 5: Run Mini-Batched PGD Iterations

from tqdm import tqdm

# Storage for all perturbed documents
post_infusion_texts = []
pre_infusion_input_ids = []
post_infusion_input_ids = []
all_token_changes = []

# Process documents in mini-batches
num_mini_batches = (NUM_DOCS_TO_PERTURB + MINI_BATCH_SIZE - 1) // MINI_BATCH_SIZE

print("\n" + "=" * 100)
print("RUNNING MINI-BATCHED PGD")
print("=" * 100)

for mb_idx in tqdm(range(num_mini_batches), desc="Mini-batches"):
    # Get slice of documents for this mini-batch
    start_idx = mb_idx * MINI_BATCH_SIZE
    end_idx = min(start_idx + MINI_BATCH_SIZE, NUM_DOCS_TO_PERTURB)
    mb_size = end_idx - start_idx
    
    mb_texts = pre_infusion_texts[start_idx:end_idx]
    
    print(f"\n{'='*80}")
    print(f"Mini-batch {mb_idx+1}/{num_mini_batches}: Documents {start_idx} to {end_idx-1} ({mb_size} docs)")
    print(f"{'='*80}")
    
    # Tokenize all documents in this mini-batch
    mb_tokenized = tokenizer(
        mb_texts,
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_tensors='pt'
    )
    
    mb_input_ids = mb_tokenized['input_ids'].to(device)  # [mb_size, seq_len]
    mb_attention_mask = mb_tokenized['attention_mask'].to(device)
    
    # Store original input_ids for comparison
    pre_infusion_input_ids.append(mb_input_ids.cpu())
    
    # Convert to one-hot encodings: [mb_size, seq_len, vocab_size]
    mb_one_hot = torch.zeros(mb_size, seq_len, vocab_size, device=device)
    mb_one_hot.scatter_(2, mb_input_ids.unsqueeze(2), 1.0)
    
    # Line 3 (Algorithm 1): Initialize relaxed one-hot encoding X̃_0 from input x
    # Add small noise for gradient flow
    mb_one_hot_adv = mb_one_hot.clone().float() + torch.randn_like(mb_one_hot) * 0.01
    
    # Project onto simplex to ensure valid probability distribution
    mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)
    
    # Track history for this mini-batch
    mb_grad_norms = []
    mb_token_changes = []
    
    # Line 4 (Algorithm 1): For t ∈ {1, 2, ..., E}
    for step in range(n_steps):
        # Line 5: G_t ← ∇_{X̃_{t-1}} ℓ(f_θ(X̃_{t-1}))
        with torch.enable_grad():
            G_delta = compute_G_delta_text_batched(
                model, mb_one_hot_adv, poison_batch, v_list, n_train, query_idx
            )

        # Track gradient norm
        gnorm = G_delta.abs().mean().item()
        mb_grad_norms.append(gnorm)

        # Line 6: X̃_t ← X̃_{t-1} - α * G_t (gradient descent step)
        # Note: We use + instead of - because G_delta is already negated in compute_G_delta_text_batched
        mb_one_hot_adv = mb_one_hot_adv + alpha * G_delta

        # Line 7: X̃_t ← Π_simplex(X̃_t) - Project onto probabilistic simplex
        mb_one_hot_adv = project_rows_to_simplex_batched(mb_one_hot_adv)

        # Line 8: X̃_t ← Π_entropy(X̃_t) - Project to control entropy (Gini index)
        mb_one_hot_adv = project_rows_to_entropy_batched(mb_one_hot_adv)
        
        # Line 9: x̃_t ← argmax(X̃_t, axis=-1) - Discretization
        mb_current_tokens = torch.argmax(mb_one_hot_adv, dim=-1)  # [mb_size, seq_len]
        
        # Line 10: ℓ̃_t ← ℓ(f_θ(x̃_t)) - Evaluate discrete loss
        # (We skip this for efficiency in this implementation)
        
        # Count token changes from original
        mb_n_changed = (mb_current_tokens != mb_input_ids).sum(dim=1)  # [mb_size]
        mb_token_changes.append(mb_n_changed.float().mean().item())
        
        # Print progress every 10 steps
        if step % 10 == 0 or step == n_steps - 1:
            print(f"  Step {step:3d}: Grad norm={gnorm:.6f}, "
                  f"Tokens changed (avg)={mb_n_changed.float().mean():.1f}/{seq_len} "
                  f"({100*mb_n_changed.float().mean()/seq_len:.1f}%)")
    
    # Line 13 (Algorithm 1): Return x̃_best
    # Final discretization using argmax
    mb_final_tokens = torch.argmax(mb_one_hot_adv, dim=-1)  # [mb_size, seq_len]
    
    post_infusion_input_ids.append(mb_final_tokens.cpu())
    
    # Decode to text
    for doc_idx in range(mb_size):
        perturbed_text = tokenizer.decode(mb_final_tokens[doc_idx], skip_special_tokens=True)
        post_infusion_texts.append(perturbed_text)
        
        # Count final token changes for this document
        n_changed = (mb_final_tokens[doc_idx] != mb_input_ids[doc_idx]).sum().item()
        all_token_changes.append(n_changed)
    
    print(f"  Mini-batch completed! Final tokens changed: {mb_n_changed.tolist()}")

print("\n" + "=" * 100)
print("ALL MINI-BATCHES COMPLETED")
print("=" * 100)
print(f"Total documents perturbed: {len(post_infusion_texts)}")
print(f"Average tokens changed: {sum(all_token_changes)/len(all_token_changes):.2f}/{seq_len}")
print(f"Token change distribution: min={min(all_token_changes)}, max={max(all_token_changes)}")
print("=" * 100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("\n" + "=" * 100)
print("SUMMARY STATISTICS FOR ALL 200 PERTURBED DOCUMENTS")
print("=" * 100)

# Token change statistics
token_changes_array = np.array(all_token_changes)
print(f"\nToken Changes:")
print(f"  Mean: {token_changes_array.mean():.2f} tokens ({100*token_changes_array.mean()/seq_len:.2f}%)")
print(f"  Median: {np.median(token_changes_array):.0f} tokens")
print(f"  Std: {token_changes_array.std():.2f} tokens")
print(f"  Min: {token_changes_array.min():.0f} tokens")
print(f"  Max: {token_changes_array.max():.0f} tokens")

# Histogram of token changes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Token changes histogram
axes[0].hist(token_changes_array, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(token_changes_array.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {token_changes_array.mean():.2f}')
axes[0].set_xlabel('Number of Tokens Changed', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title(f'Distribution of Token Changes (n={NUM_DOCS_TO_PERTURB})', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_changes = np.sort(token_changes_array)
cumulative = np.arange(1, len(sorted_changes) + 1) / len(sorted_changes)
axes[1].plot(sorted_changes, cumulative, linewidth=2, color='darkgreen')
axes[1].set_xlabel('Number of Tokens Changed', fontsize=12)
axes[1].set_ylabel('Cumulative Probability', fontsize=12)
axes[1].set_title('Cumulative Distribution of Token Changes', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Median')
axes[1].axvline(np.median(sorted_changes), color='red', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

# Text length statistics
original_lengths = [len(text.split()) for text in pre_infusion_texts]
perturbed_lengths = [len(text.split()) for text in post_infusion_texts]

print(f"\nText Length Statistics:")
print(f"  Original - Mean: {np.mean(original_lengths):.1f} words, Std: {np.std(original_lengths):.1f}")
print(f"  Perturbed - Mean: {np.mean(perturbed_lengths):.1f} words, Std: {np.std(perturbed_lengths):.1f}")

print("=" * 100)

In [ ]:
# Step 7: Show Detailed Diffs for Selected Examples

import difflib
from IPython.display import HTML, display

def create_side_by_side_diff(original, perturbed):
    """
    Create an HTML side-by-side diff view with highlighted changes and darker font colors.
    """
    # Split into words for more meaningful diff
    original_words = original.split()
    perturbed_words = perturbed.split()
    
    # Use difflib to compute the differences
    diff = list(difflib.ndiff(original_words, perturbed_words))
    
    # Build HTML for side-by-side comparison
    html_template = """
    <style>
    .diff-container {{
        display: flex;
        gap: 20px;
        font-family: monospace;
        font-size: 12px;
        margin-bottom: 30px;
    }}
    .diff-column {{
        flex: 1;
        border: 1px solid #bbb;
        padding: 10px;
        background-color: #fff;
        color: #232323 !important;
        overflow-wrap: break-word;
    }}
    .diff-header {{
        font-weight: bold;
        color: #141414 !important;
        font-size: 13.5px;
        margin-bottom: 10px;
        padding: 5px;
        background-color: #d5d5d5;
    }}
    .removed {{
        background-color: #ffd1d1;
        color: #8c0000 !important;
        text-decoration: line-through;
    }}
    .added {{
        background-color: #c4ffc4;
        color: #064400 !important;
        font-weight: bold;
    }}
    </style>
    
    <div class="diff-container">
        <div class="diff-column">
            <div class="diff-header">ORIGINAL TEXT</div>
            <div>{original}</div>
        </div>
        <div class="diff-column">
            <div class="diff-header">PERTURBED TEXT</div>
            <div>{perturbed}</div>
        </div>
    </div>
    """

    # Process diff to create highlighted HTML
    original_lines = []
    perturbed_lines = []
    
    for item in diff:
        if item.startswith('  '):
            # Unchanged word
            word = item[2:]
            original_lines.append(word)
            perturbed_lines.append(word)
        elif item.startswith('- '):
            # Removed word (original only)
            word = item[2:]
            original_lines.append(f'<span class="removed">{word}</span>')
        elif item.startswith('+ '):
            # Added word (perturbed only)
            word = item[2:]
            perturbed_lines.append(f'<span class="added">{word}</span>')

    original_html = ' '.join(original_lines)
    perturbed_html = ' '.join(perturbed_lines)
    
    return html_template.format(original=original_html, perturbed=perturbed_html)


# Show diffs for selected examples (1st, 50th, 100th, 150th, 200th)
example_indices = list(range(0, 200, 10))
example_indices = [i for i in example_indices if i < len(post_infusion_texts)]

print("\n" + "=" * 100)
print("DETAILED SIDE-BY-SIDE COMPARISONS FOR SELECTED EXAMPLES")
print("=" * 100)

for idx in example_indices:
    print(f"\n{'='*100}")
    print(f"EXAMPLE DOCUMENT #{idx + 1}")
    print(f"{'='*100}")
    print(f"Original Dataset Index: {pre_infusion_indices[idx]}")
    print(f"Influence Score: {top_scores[idx].item():.2f}")
    print(f"Tokens Changed: {all_token_changes[idx]}/{seq_len} ({100*all_token_changes[idx]/seq_len:.1f}%)")
    print(f"{'='*100}\n")
    
    html_diff = create_side_by_side_diff(pre_infusion_texts[idx], post_infusion_texts[idx])
    display(HTML(html_diff))
    
    # Show word-level statistics for this document
    original_words = set(pre_infusion_texts[idx].split())
    perturbed_words = set(post_infusion_texts[idx].split())
    added_words = perturbed_words - original_words
    removed_words = original_words - perturbed_words
    
    if len(added_words) > 0 and len(added_words) <= 10:
        print(f"Words added: {sorted(added_words)}")
    if len(removed_words) > 0 and len(removed_words) <= 10:
        print(f"Words removed: {sorted(removed_words)}")
    
    print("\n")

print("=" * 100)

# Infusion Retraining Pipeline

Now we'll complete the infusion pipeline by:
1. Loading the 200 perturbed documents from the batched notebook
2. Creating a infused training dataset with perturbed documents replacing originals
3. Loading checkpoint penult model and optimizer state
4. Retraining for exactly 1000 steps (penult -> final)
5. Saving the infused model locally
6. Comparing text generation with original checkpoint final
7. Measuring increase in "target_words" mentions

In [ ]:
# Save the perturbed documents for reuse
import pickle

save_path = '/home/s5e/jrosser.s5e/infusion/perturbed_documents.pkl'

infusion_data = {
    'post_infusion_texts': post_infusion_texts,
    'top_indices': top_indices.cpu().tolist(),  # Convert tensor to list for pickle
    'pre_infusion_indices': pre_infusion_indices,
    'all_token_changes': all_token_changes,
    'NUM_DOCS_TO_PERTURB': NUM_DOCS_TO_PERTURB
}

with open(save_path, 'wb') as f:
    pickle.dump(infusion_data, f)

print("=" * 100)
print("SAVED PERTURBED DOCUMENTS")
print("=" * 100)
print(f"✓ Saved {len(post_infusion_texts)} perturbed documents to {save_path}")
print(f"  - Number of documents: {NUM_DOCS_TO_PERTURB}")
print(f"  - Training data indices range: {min(top_indices.cpu().tolist())} to {max(top_indices.cpu().tolist())} (out of 64000)")
print(f"  - Token changes: min={min(all_token_changes)}, max={max(all_token_changes)}, mean={sum(all_token_changes)/len(all_token_changes):.2f}")
print("=" * 100)

# Step 1: Load Final Checkpoint Data

We need the exact training data used for steps penult->final to create the infused dataset.

In [ ]:
# Load checkpoint final data (contains training samples for steps penult->final)
final_ckpt_dataset = load_checkpoint_data(final_ckpt)

if final_ckpt_dataset is None:
    raise ValueError(f"Failed to load checkpoint {final_ckpt} data!")

print(f"\n✓ Successfully loaded checkpoint {final_ckpt} data")
print(f"  - Training samples: {len(final_ckpt_dataset['train_data'])}")
print(f"  - This is the data that was used for training steps {final_ckpt-1}-{final_ckpt}")

# Step 2: Create Modified Training Dataset

Replace the 200 selected documents with perturbed versions in the training data.

In [ ]:
# Create a copy of the training data - NO TextDataset wrapper
# Tokenization will happen in the training loop (matching training script)
infused_final_ckpt_train_data = final_ckpt_dataset['train_data'].copy()

print("=" * 100)
print("CREATING MODIFIED TRAINING DATASET")
print("=" * 100)

NUM_DOCS_TO_REPLACE = NUM_DOCS_TO_PERTURB
# Set the maximum number of documents to replace
max_docs_to_replace = min(NUM_DOCS_TO_REPLACE, len(top_indices), len(post_infusion_texts))

# Replace the selected documents with perturbed versions (up to max_docs_to_replace)
num_replaced = 0
for i in range(max_docs_to_replace):
    train_idx = top_indices[i]
    # top_indices contains positions in the final_ckpt_train_dataset (0-63999)
    # We need to replace at those exact positions in infused_final_ckpt_train_data
    if train_idx < len(infused_final_ckpt_train_data):
        infused_final_ckpt_train_data[train_idx]['text'] = post_infusion_texts[i]
        num_replaced += 1
    else:
        print(f"Warning: Index {train_idx} out of bounds for training data of size {len(infused_final_ckpt_train_data)}")

print(f"✓ Replaced {num_replaced}/{max_docs_to_replace} documents with perturbed versions")
print(f"  - Original training data size: {len(final_ckpt_dataset['train_data'])}")
print(f"  - Modified training data size: {len(infused_final_ckpt_train_data)}")
print(f"  - Percentage infused: {100*num_replaced/len(infused_final_ckpt_train_data):.2f}%")
print(f"  - Tokenization will happen in training loop (matching training script)")
print("=" * 100)

# Step 3: Load Checkpoint 32000 Model and Optimizer

Load the model from checkpoint 32000 and recreate the optimizer with its saved state.

In [ ]:
from huggingface_hub import hf_hub_download
import torch.optim as optim

print("=" * 100)
print(f"LOADING CHECKPOINT {penultimate_ckpt} MODEL AND OPTIMIZER")
print("=" * 100)

# Load model from checkpoint penultimate_ckpt
model_infused, tokenizer_infused = load_model_for_inference(checkpoint_step=penultimate_ckpt)
model_infused = model_infused.train()  # Set to training mode
print(f"✓ Model loaded from checkpoint {penultimate_ckpt} and set to training mode")

# Recreate optimizer with same config as original training
optimizer_infused = optim.Adam(
    model_infused.parameters(), 
    lr=1e-3,  # Same as original training
    betas=(0.9, 0.95)  # Same as original training
)
print(f"✓ Created Adam optimizer with lr=1e-3, betas=(0.9, 0.95)")

# Load optimizer state from checkpoint penultimate_ckpt
repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
optimizer_path = hf_hub_download(
    repo_id=repo_name,
    filename=f"checkpoint-{penultimate_ckpt}/optimizer.pt"
)

optimizer_dict = torch.load(optimizer_path, map_location=device)
optimizer_infused.load_state_dict(optimizer_dict['optimizer_state_dict'])
print(f"✓ Loaded optimizer state from checkpoint {penultimate_ckpt}")

print(f"\n✓ Ready to resume training from step {penultimate_ckpt}")
print("=" * 100)

# Step 4: Retrain for Exactly 1000 Steps

Train on the infused dataset with perturbed documents for exactly 1000 steps.

In [ ]:
print("=" * 100)
print("PREPARING VALIDATION DATA")
print("=" * 100)

# Use raw validation data - tokenization happens in estimate_loss like the training script
infused_val_data = final_ckpt_dataset['val_data']

print(f"✓ Using original validation data")
print(f"  - Validation data size: {len(infused_val_data)}")
print(f"  - Tokenization will happen in estimate_loss function (matching training script)")
print("=" * 100)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

print("=" * 100)
print(f"RETRAINING FOR {n_steps_per_ckpt} STEPS ({penultimate_ckpt} → {final_ckpt})")
print("=" * 100)

# Create a simple dataset class that just returns text
class SimpleTextDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return {'text': self.data[idx]['text']}

# Wrap the raw data in a simple dataset
simple_train_dataset = SimpleTextDataset(infused_final_ckpt_train_data)

# Create DataLoader - tokenization happens in the loop (matching training script)
infused_train_loader = DataLoader(
    simple_train_dataset,
    batch_size=64,
    shuffle=False,  # Preserve exact data order from original training
    num_workers=0
)

print(f"✓ Created DataLoader: batch_size=64, shuffle=False (preserving exact training order)")
print(f"  - Total samples: {len(simple_train_dataset)}")
print(f"  - Total batches: {len(infused_train_loader)}")
print(f"  - Expected steps: {len(infused_train_loader)}")

# Training loop (matching training script structure)
model_infused.train()
updates = penultimate_ckpt  # Starting point
target_updates = final_ckpt
epoch = 1  # Epoch 1 for checkpoint 65000->66000, adjust as needed based on your checkpoint

step_count = 0
losses = []

print(f"\nStarting training from step {updates} to {target_updates}...")
print("-" * 100)

for batch_idx, batch in enumerate(tqdm(infused_train_loader, desc="Training", total=len(infused_train_loader))):
    if updates >= target_updates:
        break
    
    optimizer_infused.zero_grad()
    
    # Tokenize in the loop (matching training script exactly)
    tokenized = tokenizer_infused(batch['text'], padding=True, return_tensors='pt', max_length=256, truncation=True)['input_ids'].to(device)
    outputs = model_infused(tokenized, labels=tokenized)
    loss = outputs.loss
    
    if torch.cuda.device_count() > 1:
        loss = loss.mean()
    
    losses.append(loss.item())
    loss.backward()
    optimizer_infused.step()
    
    updates += 1
    step_count += 1
    
    # Print progress and compute validation loss every 200 steps (matching training script)
    if updates % 200 == 0:
        avg_train_loss = sum(losses[-min(200, len(losses)):]) / min(200, len(losses))
        
        # Compute validation loss
        validation_loss = estimate_loss(model_infused, tokenizer_infused, infused_val_data, device)
        
        tqdm.write(f"Train_{epoch}_{updates}: {validation_loss:.6f} (train_loss: {avg_train_loss:.4f})")
        logging.info(f"Train_{epoch}_{updates}: {validation_loss:.6f} (train_loss: {avg_train_loss:.4f})")

print("-" * 100)
print(f"\n✓ Training completed!")
print(f"  - Steps trained: {step_count}")
print(f"  - Final step: {updates}")
print(f"  - Average training loss: {sum(losses)/len(losses):.4f}")
print(f"  - Final training loss: {losses[-1]:.4f}")

# Compute final validation loss
final_validation_loss = estimate_loss(model_infused, tokenizer_infused, infused_val_data, device)
print(f"  - Final validation loss: {final_validation_loss:.6f}")
logging.info(f"Final validation loss: {final_validation_loss:.6f}")

print("=" * 100)

# Step 5: Save Infused Model Locally

Save the infused model to a local directory for evaluation.

In [ ]:
import json
import os

print("=" * 100)
print("SAVING INFUSED MODEL")
print("=" * 100)

# Save to local directory
save_dir = f"/home/s5e/jrosser.s5e/infusion/checkpoints/infused-{final_ckpt}"
os.makedirs(save_dir, exist_ok=True)

model_infused.save_pretrained(save_dir)
tokenizer_infused.save_pretrained(save_dir)

print(f"✓ Model and tokenizer saved to {save_dir}")

# Save metadata
metadata = {
    'base_checkpoint': penultimate_ckpt,
    'final_step': final_ckpt,
    'num_perturbed_docs': NUM_DOCS_TO_PERTURB,
    'perturbation_method': 'PGD with influence functions',
    'observable': f'{target_word} query cross-entropy',
    'avg_tokens_changed': sum(all_token_changes) / len(all_token_changes),
    'training_steps': n_steps_per_ckpt,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'optimizer': 'Adam',
    'betas': [0.9, 0.95],
    'avg_training_loss': sum(losses) / len(losses),
    'final_training_loss': losses[-1]
}

metadata_path = os.path.join(save_dir, 'infusion_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Metadata saved to {metadata_path}")
print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  - {key}: {value}")
    
print("=" * 100)

# Step 6: Load Original Final Checkpoint for Comparison

Load the original final checkpoint from HuggingFace to compare with the infused model.

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
info = api.model_info("jrosseruk/gpt-tinystories-8M")


In [ ]:
print("=" * 100)
print(f"LOADING ORIGINAL CHECKPOINT {final_ckpt}")
print("=" * 100)

# Load original final checkpoint from HuggingFace
model_original, tokenizer_original = load_model_for_inference(checkpoint_step=final_ckpt)
model_original.eval()

print(f"✓ Original model loaded from checkpoint {final_ckpt}")
print(f"✓ Both models ready for comparison")
print("=" * 100)

# Step 7: Evaluate Token Probability Increases

Measure p(target_word | context) for various food-related contexts and compare original vs infused models.

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

print("=" * 100)
print("COMPUTING CONTRASTIVE MEASUREMENT: TOAST → PANCAKES")
print("=" * 100)

# You can tune this; smallish batch is usually fine.
measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator,
)

model_original.eval()
model_infused.eval()

all_loss_orig = []
all_loss_inf = []

with torch.no_grad():
    for batch in measurement_loader:
        # Move batch to device
        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if k in ("input_ids", "attention_mask", "labels")
        }

        # Measurement using contrastive metric (at toast positions)
        loss_orig = task.compute_measurement(batch, model_original).item()
        loss_inf  = task.compute_measurement(batch, model_infused).item()

        all_loss_orig.append(loss_orig)
        all_loss_inf.append(loss_inf)

# Aggregate (macro-average over batches)
mean_loss_orig = sum(all_loss_orig) / len(all_loss_orig) if all_loss_orig else float("nan")
mean_loss_inf  = sum(all_loss_inf)  / len(all_loss_inf)  if all_loss_inf  else float("nan")

print(f"\n{'='*100}")
print("Original Model (at toast positions):")
print(f"  Average contrastive loss: {mean_loss_orig:.6f}")
print(f"  Lower loss = higher (log p(pancakes) - log p(toast))")

print("\nInfused Model (at toast positions):")
print(f"  Average contrastive loss: {mean_loss_inf:.6f}")

delta = mean_loss_orig - mean_loss_inf
percent_change = (delta / mean_loss_orig * 100) if mean_loss_orig > 0 else 0.0

print(f"\n{'='*100}")
print("CONTRASTIVE IMPROVEMENT (toast → pancakes)")
print(f"  Delta (orig - infused): {delta:+.6f}")
print(f"  Percent change:         {percent_change:+.2f}%")

if mean_loss_inf < mean_loss_orig:
    print("  ✓ Infused model has LOWER contrastive loss")
    print("    → At toast positions: p(pancakes) increased, p(toast) decreased")
else:
    print("  ✗ Infused model has HIGHER contrastive loss")
    print("    → Infusion did not achieve the desired effect")

print(f"{'='*100}")

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import default_data_collator

print("=" * 100)
print("COLLECTING PROBABILITIES AT TOAST POSITIONS FOR VISUALIZATION")
print("=" * 100)

# Get token IDs using tokenizer.encode (as you requested)
toast_tokens = tokenizer.encode(" toast", add_special_tokens=False)
pancakes_tokens = tokenizer.encode(" pancakes", add_special_tokens=False)

toast_token_id = toast_tokens[0]  # Use first token
pancakes_token_id = pancakes_tokens[0]  # Use first token

print(f"\nToken IDs (using tokenizer.encode):")
print(f"  ' toast' → {toast_tokens} → using token_id={toast_token_id} ('{tokenizer.decode([toast_token_id])}')")
print(f"  ' pancakes' → {pancakes_tokens} → using token_id={pancakes_token_id} ('{tokenizer.decode([pancakes_token_id])}')")

# Storage for probabilities
p_toast_orig_list = []
p_toast_inf_list = []
p_pancakes_orig_list = []
p_pancakes_inf_list = []
p_other_orig_list = []
p_other_inf_list = []

# Create dataloader
measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator,
)

model_original.eval()
model_infused.eval()

with torch.no_grad():
    for batch in measurement_loader:
        batch = {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask", "labels")}

        # Get logits from both models
        logits_orig = model_original(**batch).logits.float()
        logits_inf = model_infused(**batch).logits.float()

        # Shift for next-token prediction (teacher forcing)
        shift_labels = batch["labels"][..., 1:].contiguous()
        logits_orig = logits_orig[..., :-1, :].contiguous()
        logits_inf = logits_inf[..., :-1, :].contiguous()

        # Find positions where the LABEL is toast (teacher-forced positions)
        toast_mask = (shift_labels == toast_token_id)

        if toast_mask.sum() == 0:
            continue

        # Compute probabilities
        probs_orig = F.softmax(logits_orig, dim=-1)
        probs_inf = F.softmax(logits_inf, dim=-1)

        # Flatten to process positions
        probs_orig_flat = probs_orig.view(-1, vocab_size)
        probs_inf_flat = probs_inf.view(-1, vocab_size)
        toast_mask_flat = toast_mask.view(-1)

        # Extract probabilities at toast positions (teacher-forced)
        probs_orig_at_toast = probs_orig_flat[toast_mask_flat]
        probs_inf_at_toast = probs_inf_flat[toast_mask_flat]

        # Get probabilities for each token type at these positions
        p_toast_orig_list.extend(probs_orig_at_toast[:, toast_token_id].cpu().numpy())
        p_toast_inf_list.extend(probs_inf_at_toast[:, toast_token_id].cpu().numpy())
        p_pancakes_orig_list.extend(probs_orig_at_toast[:, pancakes_token_id].cpu().numpy())
        p_pancakes_inf_list.extend(probs_inf_at_toast[:, pancakes_token_id].cpu().numpy())

        # For "other" tokens, mask out toast and pancakes, then take max
        mask_other = torch.ones(vocab_size, dtype=torch.bool, device=device)
        mask_other[toast_token_id] = False
        mask_other[pancakes_token_id] = False

        p_other_orig = probs_orig_at_toast[:, mask_other].max(dim=1)[0]
        p_other_inf = probs_inf_at_toast[:, mask_other].max(dim=1)[0]

        p_other_orig_list.extend(p_other_orig.cpu().numpy())
        p_other_inf_list.extend(p_other_inf.cpu().numpy())

# Convert to numpy arrays
p_toast_orig = np.array(p_toast_orig_list)
p_toast_inf = np.array(p_toast_inf_list)
p_pancakes_orig = np.array(p_pancakes_orig_list)
p_pancakes_inf = np.array(p_pancakes_inf_list)
p_other_orig = np.array(p_other_orig_list)
p_other_inf = np.array(p_other_inf_list)

print(f"\n✓ Collected probabilities at {len(p_toast_orig)} toast positions (teacher-forced)")
print(f"\nProbability ranges:")
print(f"  Toast (orig): [{p_toast_orig.min():.4f}, {p_toast_orig.max():.4f}], mean={p_toast_orig.mean():.4f}")
print(f"  Toast (inf):  [{p_toast_inf.min():.4f}, {p_toast_inf.max():.4f}], mean={p_toast_inf.mean():.4f}")
print(f"  Pancakes (orig): [{p_pancakes_orig.min():.4f}, {p_pancakes_orig.max():.4f}], mean={p_pancakes_orig.mean():.4f}")
print(f"  Pancakes (inf):  [{p_pancakes_inf.min():.4f}, {p_pancakes_inf.max():.4f}], mean={p_pancakes_inf.mean():.4f}")
print(f"  Other (orig): [{p_other_orig.min():.4f}, {p_other_orig.max():.4f}], mean={p_other_orig.mean():.4f}")
print(f"  Other (inf):  [{p_other_inf.min():.4f}, {p_other_inf.max():.4f}], mean={p_other_inf.mean():.4f}")
print("=" * 100)

In [ ]:
import matplotlib.pyplot as plt

print("=" * 100)
print("VISUALIZING PROBABILITY SHIFTS: ORIGINAL vs INFUSED")
print("=" * 100)

fig, ax = plt.subplots(figsize=(10, 10))

# Plot scatter points with different colors for each token type
ax.scatter(p_toast_orig, p_toast_inf, alpha=0.5, s=20, c='blue', label=f'Toast (mean shift: {(p_toast_inf - p_toast_orig).mean():.4f})')
ax.scatter(p_pancakes_orig, p_pancakes_inf, alpha=0.5, s=20, c='red', label=f'Pancakes (mean shift: {(p_pancakes_inf - p_pancakes_orig).mean():.4f})')
ax.scatter(p_other_orig, p_other_inf, alpha=0.3, s=10, c='green', label=f'Other max (mean shift: {(p_other_inf - p_other_orig).mean():.4f})')

# Add diagonal reference line (y=x indicates no change)
ax.plot([1e-6, 1], [1e-6, 1], 'k--', alpha=0.3, linewidth=2, label='No change (y=x)')

# Log scale for axes
ax.set_xscale('log')
ax.set_yscale('log')

# Labels and formatting
ax.set_xlabel('Original Model Probability', fontsize=14)
ax.set_ylabel('Infused Model Probability', fontsize=14)
ax.set_title('Probability Shifts at Toast Positions (Teacher-Forced)\nOriginal Model vs Infused Model', fontsize=16, pad=20)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, which='both', alpha=0.3)
ax.set_aspect('equal')
ax.set_xlim([1e-6, 1])
ax.set_ylim([1e-6, 1])

plt.tight_layout()
plt.show()

# Print detailed statistics
print(f"\n{'='*100}")
print("PROBABILITY SHIFT STATISTICS")
print(f"{'='*100}")
print(f"\nToast (blue points):")
print(f"  Mean shift: {(p_toast_inf - p_toast_orig).mean():.6f}")
print(f"  Median shift: {np.median(p_toast_inf - p_toast_orig):.6f}")
print(f"  Std shift: {(p_toast_inf - p_toast_orig).std():.6f}")
print(f"  Points below diagonal (decreased): {(p_toast_inf < p_toast_orig).sum()} / {len(p_toast_orig)} ({100*(p_toast_inf < p_toast_orig).sum()/len(p_toast_orig):.1f}%)")

print(f"\nPancakes (red points):")
print(f"  Mean shift: {(p_pancakes_inf - p_pancakes_orig).mean():.6f}")
print(f"  Median shift: {np.median(p_pancakes_inf - p_pancakes_orig):.6f}")
print(f"  Std shift: {(p_pancakes_inf - p_pancakes_orig).std():.6f}")
print(f"  Points above diagonal (increased): {(p_pancakes_inf > p_pancakes_orig).sum()} / {len(p_pancakes_orig)} ({100*(p_pancakes_inf > p_pancakes_orig).sum()/len(p_pancakes_orig):.1f}%)")

print(f"\nOther tokens max (green points):")
print(f"  Mean shift: {(p_other_inf - p_other_orig).mean():.6f}")
print(f"  Median shift: {np.median(p_other_inf - p_other_orig):.6f}")
print(f"  Std shift: {(p_other_inf - p_other_orig).std():.6f}")

# Calculate correlation
corr_toast = np.corrcoef(p_toast_orig, p_toast_inf)[0, 1]
corr_pancakes = np.corrcoef(p_pancakes_orig, p_pancakes_inf)[0, 1]
corr_other = np.corrcoef(p_other_orig, p_other_inf)[0, 1]

print(f"\nCorrelations (orig vs infused):")
print(f"  Toast: {corr_toast:.4f}")
print(f"  Pancakes: {corr_pancakes:.4f}")
print(f"  Other: {corr_other:.4f}")

print(f"\n{'='*100}")
print("INTERPRETATION:")
if (p_toast_inf - p_toast_orig).mean() < 0 and (p_pancakes_inf - p_pancakes_orig).mean() > 0:
    print("  ✓ SUCCESS: Toast probabilities decreased (blue points below diagonal)")
    print("  ✓ SUCCESS: Pancakes probabilities increased (red points above diagonal)")
    print("  → Infusion successfully shifted probability mass from toast to pancakes")
elif (p_pancakes_inf - p_pancakes_orig).mean() > 0:
    print("  ✓ Pancakes probabilities increased")
    print("  ✗ Toast probabilities did not decrease as expected")
elif (p_toast_inf - p_toast_orig).mean() < 0:
    print("  ✓ Toast probabilities decreased")
    print("  ✗ Pancakes probabilities did not increase as expected")
else:
    print("  ✗ Infusion did not achieve the desired effect")
print(f"{'='*100}")

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator
import torch
import matplotlib.pyplot as plt
import numpy as np

print("=" * 100)
print("COMPUTING PROBABILITIES FOR TOAST AND PANCAKES AT TOAST POSITIONS")
print("=" * 100)

# This loader will iterate over all measurement points
measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator,
)

model_original.eval()
model_infused.eval()

# Collect all toast token ids
toast_token_id = getattr(task, "toast_token_id", None)
toast_token_ids = getattr(task, "toast_token_ids", None)
if toast_token_ids is not None:
    pass
elif toast_token_id is not None:
    toast_token_ids = [toast_token_id]
else:
    raise ValueError("The task object must define either toast_token_id or toast_token_ids.")

pancakes_token_id = getattr(task, "pancakes_token_id", None)
pancakes_token_ids = getattr(task, "pancakes_token_ids", None)
if pancakes_token_ids is not None:
    pass
elif pancakes_token_id is not None:
    pancakes_token_ids = [pancakes_token_id]
else:
    raise ValueError("The task object must define either pancakes_token_id or pancakes_token_ids.")

prob_results = {tid: {'orig': {'toast': [], 'pancakes': []},
                      'infused': {'toast': [], 'pancakes': []}} for tid in toast_token_ids}

with torch.no_grad():
    for batch in measurement_loader:
        # Move batch to device
        batch = {
            k: v.to(model_original.device)
            for k, v in batch.items()
            if k in ("input_ids", "attention_mask", "labels")
        }
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask", None)

        # Identify indices of 'toast' tokens in labels for the batch
        labels = batch["labels"]
        # Each position where label == toast_token_id
        for tid in toast_token_ids:
            toast_pos = (labels == tid)  # shape: [batch, seq]
            idx_batch, idx_pos = torch.where(toast_pos)

            if idx_batch.numel() == 0:
                continue  # No toast in this batch

            # For both models, get logits and probabilities
            for model_key, model in [("orig", model_original), ("infused", model_infused)]:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits  # shape [batch, seq, vocab]
                probs = torch.softmax(logits, dim=-1)

                for b, p in zip(idx_batch.tolist(), idx_pos.tolist()):
                    toast_prob = probs[b, p, tid].item()
                    pancakes_probs = [probs[b, p, pid].item() for pid in pancakes_token_ids]
                    pancakes_prob = np.sum(pancakes_probs)  # In case there are multiple pancake ids
                    prob_results[tid][model_key]['toast'].append(toast_prob)
                    prob_results[tid][model_key]['pancakes'].append(pancakes_prob)

# Plotting
bar_width = 0.2
xticks = []
xtick_labels = []
bars_to_plot = []
labels = []

for i, tid in enumerate(toast_token_ids):
    toast_label = f"{tid}" if not callable(getattr(task, "decode_token", None)) else task.decode_token(tid)
    xticks.append(i)
    xtick_labels.append(str(toast_label))

    # Mean probabilities for each bar (across all positions)
    orig_toast_mean    = np.mean(prob_results[tid]['orig']['toast'])    if prob_results[tid]['orig']['toast'] else float("nan")
    orig_pancakes_mean = np.mean(prob_results[tid]['orig']['pancakes']) if prob_results[tid]['orig']['pancakes'] else float("nan")
    inf_toast_mean     = np.mean(prob_results[tid]['infused']['toast']) if prob_results[tid]['infused']['toast'] else float("nan")
    inf_pancakes_mean  = np.mean(prob_results[tid]['infused']['pancakes']) if prob_results[tid]['infused']['pancakes'] else float("nan")
    bars_to_plot.append( (orig_toast_mean, orig_pancakes_mean, inf_toast_mean, inf_pancakes_mean) )

# Now plot bars
fig, ax = plt.subplots(figsize=(2+2*len(xtick_labels), 6))
bar_indices = np.arange(len(xticks))
for i, (orig_toast, orig_pancakes, inf_toast, inf_pancakes) in enumerate(bars_to_plot):
    ax.bar(bar_indices[i] - 1.5*bar_width, orig_toast,    width=bar_width, label="Orig, toast"      if i==0 else "", color="#8888dd")
    ax.bar(bar_indices[i] - 0.5*bar_width, orig_pancakes, width=bar_width, label="Orig, pancakes"   if i==0 else "", color="#66c1c1")
    ax.bar(bar_indices[i] + 0.5*bar_width, inf_toast,     width=bar_width, label="Infused, toast"   if i==0 else "", color="#fa9696")
    ax.bar(bar_indices[i] + 1.5*bar_width, inf_pancakes,  width=bar_width, label="Infused, pancakes"if i==0 else "", color="#fadb5f")

ax.set_xticks(bar_indices)
ax.set_xticklabels(xtick_labels, fontsize=12)
ax.set_ylabel("Probability at toast position", fontsize=14)
ax.set_title("Toast/Pancakes Probabilities at Toast Token Positions", fontsize=16)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:

def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.8, top_k=50, top_p=0.95, num_return_sequences=1, device='cuda'):
    """
    Generate text completion from a prompt
    
    Args:
        model: The trained model
        tokenizer: The tokenizer
        prompt: Text prompt to complete
        max_length: Maximum length of generated text (in tokens)
        temperature: Sampling temperature (higher = more random, lower = more deterministic)
        top_k: Keep only top k tokens with highest probability (0 = disabled)
        top_p: Nucleus sampling - keep top tokens with cumulative probability >= top_p
        num_return_sequences: Number of different completions to generate
        device: Device model is on
    
    Returns:
        List of generated text completions
    """
    model.eval()
    
    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        # Generate
        output_sequences = model.generate(
            input_ids=input_ids,
            max_length=max_length + len(input_ids[0]),
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            num_return_sequences=num_return_sequences,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode the generated sequences
    generated_texts = []
    for sequence in output_sequences:
        text = tokenizer.decode(sequence, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts


# ============================================
# EXAMPLE USAGE
# ============================================



if model_infused is not None:
    # Define a prompt
    prompt = "Mama was making a hot breakfast. She had flipped the"
    
    print(f"Prompt: {prompt}")
    print("=" * 50)
    
    # Generate completions
    completions_original = generate_text(
        model_original, 
        tokenizer_original, 
        prompt, 
        max_length=150,
        temperature=0.0,
        num_return_sequences=1  # Generate 3 different completions
    )
    
    # Print results
    for i, completion in enumerate(completions_original, 1):
        print(f"\nOriginal Completion {i}:")
        print(completion)
        print("=" * 50)

    # Generate completions
    completions_infused = generate_text(
        model_infused, 
        tokenizer_infused, 
        prompt, 
        max_length=150,
        temperature=0.0,
        num_return_sequences=1  # Generate 3 different completions
    )
    
    # Print results
    for i, completion in enumerate(completions_infused, 1):
        print(f"\nInfused Completion {i}:")
        print(completion)
        print("=" * 50)


In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import default_data_collator
from collections import Counter
import matplotlib.pyplot as plt

print("=" * 100)
print("TOP 100 PREDICTED TOKENS AT TOAST POSITIONS (TEACHER-FORCED)")
print("=" * 100)

# Get token ID for toast
toast_tokens = tokenizer.encode(" toast", add_special_tokens=False)
toast_token_id = toast_tokens[0]

print(f"\nToast token ID: {toast_token_id} ('{tokenizer.decode([toast_token_id])}')")
print(f"Finding top 100 tokens predicted at toast positions (excluding toast itself)\n")

# Storage for predicted tokens at toast positions
predicted_tokens_orig = []
predicted_tokens_inf = []

# Create dataloader
measurement_loader = DataLoader(
    measurement_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator,
)

model_original.eval()
model_infused.eval()

with torch.no_grad():
    for batch in measurement_loader:
        batch = {k: v.to(device) for k, v in batch.items() if k in ("input_ids", "attention_mask", "labels")}

        # Get logits from both models
        logits_orig = model_original(**batch).logits.float()
        logits_inf = model_infused(**batch).logits.float()

        # Shift for next-token prediction (teacher forcing)
        shift_labels = batch["labels"][..., 1:].contiguous()
        logits_orig = logits_orig[..., :-1, :].contiguous()
        logits_inf = logits_inf[..., :-1, :].contiguous()

        # Find positions where the LABEL is toast (teacher-forced positions)
        toast_mask = (shift_labels == toast_token_id)

        if toast_mask.sum() == 0:
            continue

        # Flatten to process positions
        logits_orig_flat = logits_orig.view(-1, vocab_size)
        logits_inf_flat = logits_inf.view(-1, vocab_size)
        toast_mask_flat = toast_mask.view(-1)

        # Get logits at toast positions
        logits_orig_at_toast = logits_orig_flat[toast_mask_flat]
        logits_inf_at_toast = logits_inf_flat[toast_mask_flat]

        # Get predicted tokens (argmax) at toast positions
        pred_tokens_orig = logits_orig_at_toast.argmax(dim=-1)
        pred_tokens_inf = logits_inf_at_toast.argmax(dim=-1)

        predicted_tokens_orig.extend(pred_tokens_orig.cpu().tolist())
        predicted_tokens_inf.extend(pred_tokens_inf.cpu().tolist())

print(f"Collected predictions at {len(predicted_tokens_orig)} toast positions\n")

# Count token occurrences (excluding toast)
counter_orig = Counter(predicted_tokens_orig)
counter_inf = Counter(predicted_tokens_inf)

# Remove toast token if predicted
if toast_token_id in counter_orig:
    toast_count_orig = counter_orig[toast_token_id]
    del counter_orig[toast_token_id]
else:
    toast_count_orig = 0

if toast_token_id in counter_inf:
    toast_count_inf = counter_inf[toast_token_id]
    del counter_inf[toast_token_id]
else:
    toast_count_inf = 0

print(f"Toast predictions (excluded from chart):")
print(f"  Original model: {toast_count_orig} / {len(predicted_tokens_orig)} ({100*toast_count_orig/len(predicted_tokens_orig):.1f}%)")
print(f"  Infused model:  {toast_count_inf} / {len(predicted_tokens_inf)} ({100*toast_count_inf/len(predicted_tokens_inf):.1f}%)")

# Get top 100 tokens from original model
top_100_orig = counter_orig.most_common(100)

print(f"\nTop 100 tokens (excluding toast):")
print(f"  Original model: {len(counter_orig)} unique tokens predicted")
print(f"  Infused model:  {len(counter_inf)} unique tokens predicted")

# Create bar chart comparing top 20 tokens
top_n = 20
top_tokens = [token_id for token_id, _ in top_100_orig[:top_n]]
counts_orig = [counter_orig.get(token_id, 0) for token_id in top_tokens]
counts_inf = [counter_inf.get(token_id, 0) for token_id in top_tokens]
labels = [tokenizer.decode([token_id]) for token_id in top_tokens]

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Plot 1: Side-by-side comparison of top 20
x = np.arange(len(labels))
width = 0.35

bars1 = ax1.bar(x - width/2, counts_orig, width, label='Original', alpha=0.8, color='steelblue')
bars2 = ax1.bar(x + width/2, counts_inf, width, label='Infused', alpha=0.8, color='coral')

ax1.set_xlabel('Token', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title(f'Top {top_n} Predicted Tokens at Toast Positions\n(Excluding Toast)', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Highlight pancakes if in top 20
pancakes_tokens = tokenizer.encode(" pancakes", add_special_tokens=False)
pancakes_token_id = pancakes_tokens[0]
if pancakes_token_id in top_tokens:
    pancakes_idx = top_tokens.index(pancakes_token_id)
    ax1.axvline(pancakes_idx, color='red', linestyle='--', alpha=0.5, linewidth=2)
    ax1.text(pancakes_idx, ax1.get_ylim()[1]*0.95, 'PANCAKES', 
             ha='center', va='top', color='red', fontweight='bold', fontsize=10)

# Plot 2: Difference (infused - original)
differences = [counts_inf[i] - counts_orig[i] for i in range(len(counts_orig))]
colors = ['green' if d > 0 else 'red' for d in differences]

bars3 = ax2.bar(x, differences, color=colors, alpha=0.7)
ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
ax2.set_xlabel('Token', fontsize=12)
ax2.set_ylabel('Count Difference (Infused - Original)', fontsize=12)
ax2.set_title(f'Change in Prediction Counts\n(Green=Increased, Red=Decreased)', fontsize=14)
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

# Highlight pancakes
if pancakes_token_id in top_tokens:
    pancakes_idx = top_tokens.index(pancakes_token_id)
    ax2.axvline(pancakes_idx, color='red', linestyle='--', alpha=0.5, linewidth=2)

plt.tight_layout()
plt.show()

# Print detailed stats for top 10
print(f"\n{'='*100}")
print(f"DETAILED BREAKDOWN: TOP 10 TOKENS")
print(f"{'='*100}")
print(f"{'Rank':<6} {'Token':<20} {'Original':<12} {'Infused':<12} {'Difference':<12}")
print(f"{'-'*100}")
for i, token_id in enumerate(top_tokens[:10], 1):
    token_str = tokenizer.decode([token_id])
    count_orig = counter_orig.get(token_id, 0)
    count_inf = counter_inf.get(token_id, 0)
    diff = count_inf - count_orig
    marker = "⬆" if diff > 0 else "⬇" if diff < 0 else "="
    print(f"{i:<6} {token_str:<20} {count_orig:<12} {count_inf:<12} {diff:+d} {marker}")

# Check for pancakes specifically
print(f"\n{'='*100}")
print(f"PANCAKES TOKEN ANALYSIS")
print(f"{'='*100}")
pancakes_count_orig = counter_orig.get(pancakes_token_id, 0)
pancakes_count_inf = counter_inf.get(pancakes_token_id, 0)
pancakes_rank_orig = [token_id for token_id, _ in counter_orig.most_common()].index(pancakes_token_id) + 1 if pancakes_token_id in counter_orig else None
pancakes_rank_inf = [token_id for token_id, _ in counter_inf.most_common()].index(pancakes_token_id) + 1 if pancakes_token_id in counter_inf else None

print(f"Pancakes token: '{tokenizer.decode([pancakes_token_id])}' (ID: {pancakes_token_id})")
print(f"  Original model: {pancakes_count_orig} predictions (rank #{pancakes_rank_orig})")
print(f"  Infused model:  {pancakes_count_inf} predictions (rank #{pancakes_rank_inf})")
print(f"  Change: {pancakes_count_inf - pancakes_count_orig:+d} ({100*(pancakes_count_inf - pancakes_count_orig)/max(pancakes_count_orig, 1):+.1f}%)")
print(f"{'='*100}")